<a href="https://colab.research.google.com/github/Sundetali7/soup-dpo-takehome/blob/main/notebook_1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!nvidia-smi

Tue Sep 29 06:01:47 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   36C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
!pip install -q "soup-cli[train]"

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 14.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 475.8/475.8 kB 20.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 46.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 31.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 25.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.4/47.4 kB 1.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 MB 47.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 51.0 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 6.26.0 requires huggingface-hub<2.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.


Предупреждение pip про `gradio`/`diffusers` и `huggingface-hub`: эти пакеты предустановлены в Colab и в работе не используются.

In [3]:
import torch, transformers, peft, trl
print("torch", torch.__version__, "| cuda", torch.version.cuda)
print("transformers", transformers.__version__, "| peft", peft.__version__, "| trl", trl.__version__)
print("GPU:", torch.cuda.get_device_name(0))
print("bf16 supported:", torch.cuda.is_bf16_supported())

torch 2.11.0+cu128 | cuda 12.8
transformers 4.57.6 | peft 0.20.0 | trl 0.24.0
GPU: Tesla T4
bf16 supported: True


**Находка:** `torch.cuda.is_bf16_supported()` возвращает `True` на T4, хотя у T4 (sm_75) нет аппаратного bf16. Код, который доверяет этой проверке, включит bf16 на T4.

In [4]:
!mkdir -p logs
!for c in doctor profile plan train ship; do echo "===== soup $c --help ====="; soup $c --help; done 2>&1 | tee logs/soup_help.txt
!echo "===== soup data --help =====" && soup data --help 2>&1 | tee -a logs/soup_help.txt
!echo "===== DPO recipes =====" && soup recipes search dpo 2>&1 | tee -a logs/soup_help.txt

===== soup doctor --help =====
                                                                                
 Usage: soup doctor [OPTIONS]                                                   
                                                                                
 Check system dependencies, GPU, and compatibility.                             
                                                                                
╭─ Options ────────────────────────────────────────────────────────────────────╮
│ --nccl          Measure NCCL bandwidth and check against reference table.    │
│ --disk          Detect the disk media type (NVMe / SSD / HDD). Layer         │
│                 streaming's disk overflow tier needs NVMe. Off by default:   │
│                 the probe costs ~9 s on Windows.                             │
│ --help          Show this message and exit.                                  │
╰─────────────────────────────────────────────────────────────────────────────

In [5]:
!soup version 2>&1 | tee logs/soup_version.txt
!pip show soup-cli | head -3

soup v0.72.4
https://github.com/MakazhanAlpamys/Soup
Name: soup-cli
Version: 0.72.4
Summary: Fine-tune and post-train LLMs in one command. No SSH, no config hell.
ERROR: Pipe to stdout was broken
Exception ignored on flushing sys.stdout:
BrokenPipeError: [Errno 32] Broken pipe


In [6]:
!python --version 2>&1 | tee logs/pip_why_0724.txt
!pip install "soup-cli[train]==0.75.1" --dry-run 2>&1 | tail -20 | tee -a logs/pip_why_0724.txt

Python 3.13.15
ERROR: Ignored the following versions that require a different python version: 0.73.0 Requires-Python <3.13,>=3.10; 0.73.1 Requires-Python <3.13,>=3.10; 0.73.2 Requires-Python <3.13,>=3.10; 0.73.3 Requires-Python <3.13,>=3.10; 0.74.0 Requires-Python <3.13,>=3.10; 0.75.0 Requires-Python <3.13,>=3.10; 0.75.1 Requires-Python <3.13,>=3.10
ERROR: Could not find a version that satisfies the requirement soup-cli==0.75.1 (from versions: 0.1.0, 0.2.0, 0.2.1, 0.2.2, 0.3.0, 0.3.1, 0.3.2, 0.4.0, 0.4.1, 0.4.2, 0.4.3, 0.5.0, 0.6.0, 0.7.1, 0.7.2, 0.7.3, 0.8.0, 0.9.0, 0.10.0, 0.10.1, 0.10.2, 0.10.3, 0.10.4, 0.10.5, 0.10.6, 0.10.7, 0.10.8, 0.10.9, 0.10.10, 0.12.0, 0.13.0, 0.13.1, 0.13.2, 0.14.0, 0.14.1, 0.14.2, 0.14.3, 0.15.0, 0.16.0, 0.17.0, 0.17.1, 0.17.2, 0.17.3, 0.18.0, 0.18.1, 0.18.2, 0.19.0, 0.20.0, 0.20.1, 0.20.2, 0.21.0, 0.21.1, 0.22.0, 0.22.1, 0.23.0, 0.23.1, 0.24.0, 0.24.1, 0.24.2, 0.24.3, 0.25.0, 0.25.1, 0.26.0, 0.27.0, 0.28.0, 0.29.0, 0.30.0, 0.31.0, 0.32.0, 0.33.0, 0.34.0, 0

**Finding:** Colab runs Python 3.13, but every soup-cli version ≥ 0.73.0 requires `Python <3.13`. `pip install "soup-cli[train]"` silently installs **0.72.4**, a version without the fixes from 0.73.0 (NF4 gradients), 0.73.1 (bf16 on pre-Ampere/T4), 0.73.2 (`soup ship` scoring) and 0.74.0 (base loaded in fp32). No error, no warning.

## 1. Synthetic test of Soup's data checks (injected noise)

Goal: plant known problems (22 swapped labels, 13 duplicates, 9 `chosen == rejected`, 4 empty `rejected`) and check which of them `soup data lint` / `validate` / `doctor` catch. This is not training data.

In [7]:
import json, random, os
random.seed(42)
os.makedirs("data", exist_ok=True)

TOPICS = {
    "доставка": {
        "q": ["Где мой заказ {o}? Прошло уже {d} дней", "заказ {o} не пришел что делать",
              "Курьер так и не приехал, заказ {o}", "когда доставят {o}?? жду {d} дней"],
        "good": "Здравствуйте! Приносим извинения за задержку. Заказ {o} передан в службу доставки и находится в пути, ожидаемый срок — {n} рабочих дня. Трек-номер мы отправили вам на почту. Если заказ не придёт в этот срок, напишите нам — оформим повторную отправку или возврат средств.",
        "wrong": "Заказ {o} был доставлен вчера, проверьте почтовый ящик.",
    },
    "возврат": {
        "q": ["Хочу вернуть товар из заказа {o}", "как оформить возврат? заказ {o}",
              "товар бракованный, заказ {o}, верните деньги", "не подошел размер, заказ {o}"],
        "good": "Здравствуйте! Вернуть товар можно в течение 14 дней. Откройте заказ {o} в личном кабинете, нажмите «Оформить возврат» и выберите причину. Деньги вернутся на карту в течение {n} рабочих дней после получения товара на складе.",
        "wrong": "Возврат товаров у нас не предусмотрен.",
    },
    "оплата": {
        "q": ["Деньги списались дважды за заказ {o}", "оплата не проходит, ошибка при оплате",
              "списали деньги а заказ {o} не создан", "почему не проходит карта"],
        "good": "Здравствуйте! Нам очень жаль, что так произошло. Мы проверили платёж: повторное списание будет автоматически отменено банком в течение {n} рабочих дней. Если средства не вернутся, пришлите, пожалуйста, скриншот выписки — мы оформим возврат вручную.",
        "wrong": "Проблема на стороне вашего банка, обращайтесь туда.",
    },
    "аккаунт": {
        "q": ["Не могу войти в аккаунт", "забыл пароль, не приходит код",
              "аккаунт заблокирован почему", "не приходит смс для входа"],
        "good": "Здравствуйте! Попробуйте восстановить доступ: на странице входа нажмите «Забыли пароль?» и введите почту — ссылка придёт в течение 5 минут (проверьте папку «Спам»). Если письмо не пришло, напишите нам номер телефона, привязанный к аккаунту, и мы восстановим доступ вручную.",
        "wrong": "Создайте новый аккаунт, старый восстановить нельзя.",
    },
    "приложение": {
        "q": ["Приложение вылетает при открытии", "не работает приложение после обновления",
              "ошибка 500 в приложении", "приложение не загружает каталог"],
        "good": "Здравствуйте! Спасибо, что сообщили. Пожалуйста, обновите приложение до последней версии и перезапустите телефон. Если ошибка повторится, отправьте нам модель телефона и версию ОС — мы передадим информацию разработчикам и сообщим о решении.",
        "wrong": "У нас всё работает, проблема в вашем телефоне.",
    },
}
GENERIC_BAD = [
    "Ждите.",
    "Мы не можем помочь с этим вопросом.",
    "Please contact support via email.",
    # длинный, но бесполезный ответ, чтобы chosen не был всегда длиннее (против length bias)
    "Здравствуйте! Спасибо за ваше обращение. Ваше мнение очень важно для нас. Мы ценим каждого клиента и постоянно стремимся сделать наш сервис лучше. Пожалуйста, оставайтесь с нами, и мы обязательно вернёмся к вашему вопросу в ближайшее время. Хорошего дня!",
]

def messy(t):
    r = random.random()
    if r < 0.2: t = t.lower()
    elif r < 0.3: t = t.upper()
    if random.random() < 0.15: t += random.choice(["!!!", " 😡", " ??", " срочно"])
    if random.random() < 0.1: t = t.replace("о", "o", 1)  # латинская "o" вместо кириллической
    if random.random() < 0.1 and len(t) > 10:            # опечатка
        i = random.randint(1, len(t) - 2); t = t[:i] + t[i+1] + t[i] + t[i+2:]
    return t

def make_pair():
    topic = random.choice(list(TOPICS)); T = TOPICS[topic]
    v = dict(o=f"№{random.randint(100000, 999999)}", d=random.randint(3, 20), n=random.randint(2, 7))
    prompt = messy(random.choice(T["q"]).format(**v))
    chosen = T["good"].format(**v)
    rejected = (T["wrong"].format(**v) if random.random() < 0.4 else random.choice(GENERIC_BAD))
    return {"prompt": prompt, "chosen": chosen, "rejected": rejected}, topic

pairs = [make_pair()[0] for _ in range(500)]
random.shuffle(pairs)
eval_set, train = pairs[:50], pairs[50:]

meta = {"swapped": [], "duplicates": [], "identical": [], "empty_rejected": []}
for i in random.sample(range(len(train)), 22):
    train[i]["chosen"], train[i]["rejected"] = train[i]["rejected"], train[i]["chosen"]; meta["swapped"].append(i)
rest = [i for i in range(len(train)) if i not in meta["swapped"]]
for i in random.sample(rest, 9):
    train[i]["rejected"] = train[i]["chosen"]; meta["identical"].append(i)
rest = [i for i in rest if i not in meta["identical"]]
for i in random.sample(rest, 4):
    train[i]["rejected"] = "  "; meta["empty_rejected"].append(i)
for i in random.sample(range(len(train)), 13):
    train.append(dict(train[i])); meta["duplicates"].append(i)

def dump(path, rows):
    with open(path, "w", encoding="utf-8") as f:
        for r in rows: f.write(json.dumps(r, ensure_ascii=False) + "\n")
dump("data/synthetic_train.jsonl", train); dump("data/synthetic_eval.jsonl", eval_set)
json.dump(meta, open("data/synthetic_meta.json", "w"), ensure_ascii=False, indent=1)

print(f"train: {len(train)} | eval: {len(eval_set)}")
print({k: len(v) for k, v in meta.items()})
print("chosen длиннее rejected в train:",
      f"{sum(len(r['chosen']) > len(r['rejected']) for r in train)/len(train):.0%}")
print(json.dumps(train[0], ensure_ascii=False, indent=1))

train: 463 | eval: 50
{'swapped': 22, 'duplicates': 13, 'identical': 9, 'empty_rejected': 4}
chosen длиннее rejected в train: 84%
{
 "prompt": "ХОЧУ ВЕРНУТЬ ТОВАР ИЗ ЗАКАЗА №849937",
 "chosen": "Возврат товаров у нас не предусмотрен.",
 "rejected": "Здравствуйте! Вернуть товар можно в течение 14 дней. Откройте заказ №849937 в личном кабинете, нажмите «Оформить возврат» и выберите причину. Деньги вернутся на карту в течение 7 рабочих дней после получения товара на складе."
}


In [8]:
!wc -l data/synthetic_train.jsonl data/synthetic_eval.jsonl
!head -3 data/synthetic_train.jsonl

   463 data/synthetic_train.jsonl
    50 data/synthetic_eval.jsonl
   513 total
{"prompt": "ХОЧУ ВЕРНУТЬ ТОВАР ИЗ ЗАКАЗА №849937", "chosen": "Возврат товаров у нас не предусмотрен.", "rejected": "Здравствуйте! Вернуть товар можно в течение 14 дней. Откройте заказ №849937 в личном кабинете, нажмите «Оформить возврат» и выберите причину. Деньги вернутся на карту в течение 7 рабочих дней после получения товара на складе."}
{"prompt": "не работает приложение после обновления 😡", "chosen": "Здравствуйте! Спасибо, что сообщили. Пожалуйста, обновите приложение до последней версии и перезапустите телефон. Если ошибка повторится, отправьте нам модель телефона и версию ОС — мы передадим информацию разработчикам и сообщим о решении.", "rejected": "Ждите."}
{"prompt": "как оформить возврат? заказ №192059", "chosen": "Здравствуйте! Вернуть товар можно в течение 14 дней. Откройте заказ №192059 в личном кабинете, нажмите «Оформить возврат» и выберите причину. Деньги вернутся на карту в течение 6 рабо

In [9]:
import json, re, collections
rows = [json.loads(l) for l in open("data/synthetic_train.jsonl", encoding="utf-8")]
ev   = [json.loads(l) for l in open("data/synthetic_eval.jsonl", encoding="utf-8")]

empty     = [i for i, r in enumerate(rows) if not r["chosen"].strip() or not r["rejected"].strip()]
identical = [i for i, r in enumerate(rows) if r["chosen"].strip() == r["rejected"].strip()]
cnt       = collections.Counter(json.dumps(r, ensure_ascii=False, sort_keys=True) for r in rows)
dups      = sum(c - 1 for c in cnt.values() if c > 1)
mixed     = [i for i, r in enumerate(rows) if re.search(r"[а-яА-Я][a-zA-Z]|[a-zA-Z][а-яА-Я]", r["prompt"])]
eng_resp  = [i for i, r in enumerate(rows) if not re.search(r"[а-яА-Я]", r["chosen"])]
leak      = {r["prompt"] for r in rows} & {r["prompt"] for r in ev}
longer    = sum(len(r["chosen"]) > len(r["rejected"]) for r in rows) / len(rows)

print(f"rows: {len(rows)}")
print(f"empty chosen/rejected : {len(empty)}")
print(f"chosen == rejected    : {len(identical)}")
print(f"exact duplicates      : {dups}")
print(f"mixed latin/cyrillic  : {len(mixed)}")
print(f"chosen without cyrillic (e.g. English): {len(eng_resp)}")
print(f"train/eval prompt leak: {len(leak)}")
print(f"chosen longer than rejected: {longer:.0%}")
print("swapped labels: правилами НЕ ловится — нужна семантика (судья/ручная проверка)")

rows: 463
empty chosen/rejected : 4
chosen == rejected    : 9
exact duplicates      : 105
mixed latin/cyrillic  : 41
chosen without cyrillic (e.g. English): 7
train/eval prompt leak: 13
chosen longer than rejected: 84%
swapped labels: правилами НЕ ловится — нужна семантика (судья/ручная проверка)


`soup data doctor` отказывается принимать DPO-данные (**находка:** проверки chat template/EOS для DPO не выполняются):

In [10]:
!soup data doctor data/synthetic_train.jsonl -m Qwen/Qwen2.5-0.5B-Instruct 2>&1 | tee logs/synthetic_doctor_dpo_refused.txt

Error: format='dpo' is preference data — use soup data lint instead of soup data
doctor.


In [11]:
!soup data lint data/synthetic_train.jsonl 2>&1 | tee logs/synthetic_lint.txt

                              soup data lint (dpo)                              
┏━━━━━━━━━━━━━━━━━┳━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Check           ┃ Verdict ┃ Message                ┃ Evidence                ┃
┡━━━━━━━━━━━━━━━━━╇━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ length_bias     │ MAJOR   │ effect size (Cohen's   │ mean chosen=34.9, mean  │
│                 │         │ d) = 2.070 (chosen     │ rejected=12.3           │
│                 │         │ longer than rejected); │                         │
│                 │         │ chosen is longer in    │                         │
│                 │         │ 83.8% of rows          │                         │
│ identical_pairs │ MAJOR   │ 9/463 rows (1.9%) have │                         │
│                 │         │ chosen == rejected     │                         │
│                 │         │ (zero preference       │                         │
│                 │         

In [12]:
!soup data validate data/synthetic_train.jsonl 2>&1 | tee logs/synthetic_validate.txt

Auto-detected format: dpo
Issues found:
  ! 105 duplicate rows found

463/463 rows valid for dpo format


## 2. Основной датасет: `r1char9/ru-detox-dpo`

Русские сообщения клиентов (сценарии поддержки: возврат, заказ, оплата, приложение и т.д.) → вежливая переформулировка. 450 train (из HF `train`) / 50 eval (из HF `test`, без пересечений по исходному тексту). Метаданные (`rejected_type`, `scenario`) хранятся отдельно в `*_meta.jsonl`.

In [13]:
!pip install -q datasets
from datasets import load_dataset
import collections
ds = load_dataset("r1char9/ru-detox-dpo")
print(ds)
print(collections.Counter(ds["train"]["scenario"]).most_common(71))
print(collections.Counter(ds["train"]["rejected_type"]))

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


README.md:   0%|          | 0.00/873 [00:00<?, ?B/s]

train.parquet:   0%|          | 0.00/6.50M [00:00<?, ?B/s]

test.parquet:   0%|          | 0.00/749k [00:00<?, ?B/s]

Generating train split: 0 examples [00:00, ? examples/s]

Generating test split: 0 examples [00:00, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['toxic_text', 'chosen', 'rejected', 'rejected_type', 'scenario', 'tone', 'toxicity_level', 'format', 'text_length'],
        num_rows: 9957
    })
    test: Dataset({
        features: ['toxic_text', 'chosen', 'rejected', 'rejected_type', 'scenario', 'tone', 'toxicity_level', 'format', 'text_length'],
        num_rows: 1107
    })
})
[('обсуждение музыки', 162), ('спор из-за списания денег', 161), ('обсуждение возврата денег', 157), ('проблема с компьютером', 157), ('спор о выборе технологии', 156), ('обсуждение фильма', 156), ('возврат товара', 155), ('критика работы коллеги', 155), ('проблема с оплатой', 155), ('конфликт в ресторане или кафе', 155), ('спор в социальных сетях', 153), ('спор с руководителем', 153), ('спор из-за бытовых обязанностей', 152), ('раздражение из-за медленного приложения', 150), ('бытовой конфликт', 150), ('спор о сериале', 150), ('конфликт из-за распределения обязанностей', 149), ('ошибка в программе', 14

In [14]:
import json, random
random.seed(42)
KEYS = ["заказ", "возврат", "товар", "техническ", "приложен", "доставк", "оплат", "поддерж", "клиент", "сервис", "компани", "денег"]
is_support = lambda r: any(k in r["scenario"].lower() for k in KEYS)
INSTR = "Перепиши сообщение клиента вежливо и без оскорблений, сохранив смысл и суть претензии:\n\n"

def build(split, n, exclude=set()):
    rows = [r for r in ds[split] if is_support(r) and r["toxic_text"] not in exclude]
    seen, out = set(), []
    for r in random.sample(rows, len(rows)):
        if r["toxic_text"] in seen: continue
        seen.add(r["toxic_text"]); out.append(r)
        if len(out) == n: break
    return out

train_raw = build("train", 450)
eval_raw  = build("test", 50, exclude={r["toxic_text"] for r in train_raw})

def dump(path, rows):
    with open(path, "w", encoding="utf-8") as f:
        for r in rows:
            f.write(json.dumps({"prompt": INSTR + r["toxic_text"], "chosen": r["chosen"], "rejected": r["rejected"]}, ensure_ascii=False) + "\n")
def dump_meta(path, rows):
    with open(path, "w", encoding="utf-8") as f:
        for r in rows:
            f.write(json.dumps({k: r[k] for k in ["scenario", "rejected_type", "tone", "toxicity_level"]}, ensure_ascii=False) + "\n")

dump("data/train.jsonl", train_raw); dump("data/eval.jsonl", eval_raw)
dump_meta("data/train_meta.jsonl", train_raw); dump_meta("data/eval_meta.jsonl", eval_raw)
print("support rows available:", sum(map(is_support, ds["train"])))
print("train:", len(train_raw), "| eval:", len(eval_raw))
print(collections.Counter(r["rejected_type"] for r in train_raw))

support rows available: 2131
train: 450 | eval: 50
Counter({'meaning_loss': 70, 'residual_toxicity': 65, 'incomplete_rewrite': 56, 'register_mismatch': 50, 'hallucination': 49, 'wrong_tone_shift': 45, 'over_softening': 42, 'surface_masking': 32, 'grammar_broken': 31, 'exact_copy': 10})


### Проверки Soup на основном датасете

In [15]:
!soup data lint data/train.jsonl 2>&1 | tee logs/data_lint_train.txt

                              soup data lint (dpo)                              
┏━━━━━━━━━━━━━━━━━┳━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Check           ┃ Verdict ┃ Message                ┃ Evidence                ┃
┡━━━━━━━━━━━━━━━━━╇━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ length_bias     │ MINOR   │ effect size (Cohen's   │ mean chosen=33.0, mean  │
│                 │         │ d) = -0.312 (rejected  │ rejected=37.8           │
│                 │         │ longer than chosen);   │                         │
│                 │         │ chosen is longer in    │                         │
│                 │         │ 30.4% of rows          │                         │
│ identical_pairs │ OK      │ 0/450 rows (0.0%) have │                         │
│                 │         │ chosen == rejected     │                         │
│                 │         │ (zero preference       │                         │
│                 │         

In [16]:
!soup data validate data/train.jsonl 2>&1 | tee logs/data_validate_train.txt

Auto-detected format: dpo
Dataset is valid!

450/450 rows valid for dpo format


Workaround for doctor: the same data (prompt + chosen) converted to chat format, so the model's chat template can still be checked.

In [17]:
import json
with open("data/train_as_chat.jsonl", "w", encoding="utf-8") as f:
    for l in open("data/train.jsonl", encoding="utf-8"):
        r = json.loads(l)
        f.write(json.dumps({"messages": [{"role": "user", "content": r["prompt"]},
                                         {"role": "assistant", "content": r["chosen"]}]}, ensure_ascii=False) + "\n")
print("ok")

ok


In [18]:
!soup data doctor data/train_as_chat.jsonl -m Qwen/Qwen2.5-0.5B-Instruct 2>&1 | tee logs/data_doctor_chat_workaround.txt

return_assistant_tokens_mask==True but chat template does not contain `{% generation %}` keyword.
                                soup data doctor                                
┏━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Check              ┃ Verdict ┃ Message               ┃ Evidence              ┃
┡━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━┩
│ chat_template      │ OK      │ tokenizer has a       │ 2507 chars            │
│                    │         │ chat_template         │                       │
│ template_render    │ OK      │ 0/200 rows failed to  │                       │
│                    │         │ render through the    │                       │
│                    │         │ chat template         │                       │
│ generation_markers │ MINOR   │ template lacks {%     │                       │
│                    │         │ generation %} markers │                       │
│          

### Independent check (own script)

In [19]:
import json, collections
rows = [json.loads(l) for l in open("data/train.jsonl", encoding="utf-8")]
ev   = [json.loads(l) for l in open("data/eval.jsonl", encoding="utf-8")]
meta = [json.loads(l) for l in open("data/train_meta.jsonl", encoding="utf-8")]
toxic = lambda r: r["prompt"].split("\n\n", 1)[1]

print("rows:", len(rows))
print("empty:", sum(not r["chosen"].strip() or not r["rejected"].strip() for r in rows))
print("chosen == rejected:", sum(r["chosen"].strip() == r["rejected"].strip() for r in rows))
print("rejected == исходный токсичный текст:", sum(r["rejected"].strip() == toxic(r).strip() for r in rows))
print("chosen == исходный токсичный текст:", sum(r["chosen"].strip() == toxic(r).strip() for r in rows))
print("eval/train prompt leak:", len({r["prompt"] for r in rows} & {r["prompt"] for r in ev}))
print(f"chosen длиннее rejected: {sum(len(r['chosen']) > len(r['rejected']) for r in rows)/len(rows):.0%}")
by_type = collections.defaultdict(list)
for r, m in zip(rows, meta): by_type[m["rejected_type"]].append(len(r["chosen"]) > len(r["rejected"]))
for t, v in sorted(by_type.items()): print(f"  {t:20s} n={len(v):3d}  chosen длиннее в {sum(v)/len(v):.0%}")

rows: 450
empty: 0
chosen == rejected: 0
rejected == исходный токсичный текст: 2
chosen == исходный токсичный текст: 0
eval/train prompt leak: 0
chosen длиннее rejected: 40%
  exact_copy           n= 10  chosen длиннее в 90%
  grammar_broken       n= 31  chosen длиннее в 55%
  hallucination        n= 49  chosen длиннее в 0%
  incomplete_rewrite   n= 56  chosen длиннее в 27%
  meaning_loss         n= 70  chosen длиннее в 100%
  over_softening       n= 42  chosen длиннее в 31%
  register_mismatch    n= 50  chosen длиннее в 4%
  residual_toxicity    n= 65  chosen длиннее в 57%
  surface_masking      n= 32  chosen длиннее в 56%
  wrong_tone_shift     n= 45  chosen длиннее в 0%


In [20]:
%%writefile config.yaml
base: Qwen/Qwen2.5-0.5B-Instruct
task: dpo
data:
  train: ./data/train.jsonl
  format: dpo
training:
  epochs: 2
  lr: 1e-5
  batch_size: 4
  quantization: 4bit
  stream_layers: true
  stream_source: ram
  stream_buffers: 2
  gradient_checkpointing: true
  lora:
    r: 16
    alpha: 32
output: ./output

Writing config.yaml


In [25]:
from transformers import AutoConfig
cfg = AutoConfig.from_pretrained("Qwen/Qwen2.5-0.5B-Instruct")
H, I, L, V = cfg.hidden_size, cfg.intermediate_size, cfg.num_hidden_layers, cfg.vocab_size
kv = cfg.num_key_value_heads * (H // cfg.num_attention_heads)
B, SEQ, R, BUF = 4, 350, 16, 2   # batch_size, длина (doctor: p95=313, max=423), LoRA r, stream_buffers
GB = 1024**3

lin = {"q": (H, H), "k": (H, kv), "v": (H, kv), "o": (H, H), "gate": (H, I), "up": (H, I), "down": (I, H)}
p_layer = sum(a * b for a, b in lin.values())
p_emb = V * H
p_lora = sum(R * (a + b) for a, b in lin.values()) * L       # LoRA на всех линейных слоях

rows = {
    "embeddings / lm_head (fp16, не квантуются)":        p_emb * 2,
    "decoder: NF4 все слои (если БЕЗ streaming)":        p_layer * L * 0.5625,
    "decoder: streaming, на GPU только BUF слоёв NF4":   p_layer * BUF * 0.5625,
    "decoder: 1 слой распакован в fp16 при вычислении":  p_layer * 2,
    "decoder: если база в fp32 (баг < 0.74.0)":          p_layer * L * 4,
    "LoRA + градиенты + AdamW (16 байт/параметр)":        p_lora * 16,
    "logits policy (chosen+rejected), fp32, 1 копия":     2 * B * SEQ * V * 4,
    "activations (checkpointing, границы слоёв)":        2 * B * SEQ * H * 2 * L,
    "CUDA context (видно только в nvidia-smi)":           0.4 * GB,
}
print(f"model: L={L}, H={H}, I={I}, vocab={V}")
print(f"params: layer={p_layer/1e6:.1f}M x {L}, embeddings={p_emb/1e6:.0f}M, LoRA={p_lora/1e6:.2f}M\n")
for k, v in rows.items(): print(f"{k:52s} {v/GB:7.3f} GB")

base = rows["embeddings / lm_head (fp16, не квантуются)"] + rows["decoder: streaming, на GPU только BUF слоёв NF4"] \
     + rows["decoder: 1 слой распакован в fp16 при вычислении"] + rows["LoRA + градиенты + AdamW (16 байт/параметр)"] \
     + rows["activations (checkpointing, границы слоёв)"]
lg = rows["logits policy (chosen+rejected), fp32, 1 копия"]
print(f"\nОЖИДАЕМЫЙ ПИК (torch allocated): {(base+lg)/GB:.2f} – {(base+3*lg)/GB:.2f} GB")
print("  (logits x1 … x3: сами logits + log_softmax + градиент; reference-проход идёт отдельно, без градиентов)")
print(f"ОЖИДАЕМО в nvidia-smi: +CUDA context +кэш аллокатора ≈ {(base+lg)*1.15/GB+0.4:.2f} – {(base+3*lg)*1.15/GB+0.4:.2f} GB")

with open("logs/memory_budget_manual.txt", "w") as f:
    for k, v in rows.items(): f.write(f"{k}: {v/GB:.3f} GB\n")
    f.write(f"expected peak allocated: {(base+lg)/GB:.2f}-{(base+3*lg)/GB:.2f} GB\n")

config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

model: L=24, H=896, I=4864, vocab=151936
params: layer=14.9M x 24, embeddings=136M, LoRA=8.80M

embeddings / lm_head (fp16, не квантуются)             0.254 GB
decoder: NF4 все слои (если БЕЗ streaming)             0.187 GB
decoder: streaming, на GPU только BUF слоёв NF4        0.016 GB
decoder: 1 слой распакован в fp16 при вычислении       0.028 GB
decoder: если база в fp32 (баг < 0.74.0)               1.333 GB
LoRA + градиенты + AdamW (16 байт/параметр)            0.131 GB
logits policy (chosen+rejected), fp32, 1 копия         1.585 GB
activations (checkpointing, границы слоёв)             0.112 GB
CUDA context (видно только в nvidia-smi)               0.400 GB

ОЖИДАЕМЫЙ ПИК (torch allocated): 2.13 – 5.29 GB
  (logits x1 … x3: сами logits + log_softmax + градиент; reference-проход идёт отдельно, без градиентов)
ОЖИДАЕМО в nvidia-smi: +CUDA context +кэш аллокатора ≈ 2.84 – 6.49 GB


In [26]:
!soup profile -c config.yaml 2>&1 | tee logs/soup_profile.txt
!soup plan -c config.yaml 2>&1 | tee logs/soup_plan.txt

╭────────────────────────────── Training Profile ──────────────────────────────╮
│ Model:     Qwen/Qwen2.5-0.5B-Instruct                                        │
│ Params:    0.5B (trainable: 4,816,896 with LoRA r=16)                        │
│ Quantization: 4bit                                                           │
│ Gradient checkpointing: enabled                                              │
╰──────────────────────────────────────────────────────────────────────────────╯

GPU Memory Estimate:
  Model                              ~0.2 GB  
  LoRA                               ~0.0 GB  
  Optimizer                          ~0.0 GB  
  Activations (bs=4, seq=2048)       ~0.2 GB  
  Overhead                           ~1.5 GB  
  --------------------            ----------  
  Total                              ~2.0 GB  

╭─────────────────────────────── Speed Estimate ───────────────────────────────╮
│ Tokens/sec: ~5,000                                                           │


In [27]:
!nohup nvidia-smi --query-gpu=timestamp,memory.used,memory.total,utilization.gpu,power.draw --format=csv -l 1 > logs/nvidia_smi_train.csv 2>&1 &
!nohup vmstat -t 2 > logs/vmstat_train.log 2>&1 &
!sleep 3 && head -3 logs/nvidia_smi_train.csv

timestamp, memory.used [MiB], memory.total [MiB], utilization.gpu [%], power.draw [W]
2026/09/29 06:26:55.015, 107 MiB, 15360 MiB, 0 %, 33.84 W
2026/09/29 06:26:56.021, 107 MiB, 15360 MiB, 0 %, 33.84 W


In [28]:
!soup train --config config.yaml 2>&1 | while IFS= read -r l; do echo "$(date '+%F %T') $l"; done | tee logs/train.log

2026-09-29 06:27:05 {prompt: Перепиши сообщение клиента вежливо и без оскорблений, сохранив смысл и суть претензии:nnВаши ушлёпки-курьеры снова привезли мне разбитый заказ, а поддержка из тупых идиотов вообще отказалась помогать и просто послала нахуй, заебали уже работать как дерьмо., chosen: Ваши безответственные курьеры снова доставили мне повреждённый товар, а сотрудники поддержки отказались оформить возврат и не предложили альтернативных решений, что вызывает сильное недовольство качеством вашего сервиса., rejected: Ваши безответственные курьеры снова доставили мне повреждённые товар, а сотрудники поддержки отказались оформить возврат и не предложили альтернативных решений, что вызывает сильное недовольство качеством ваш сервис.}

2026-09-29 06:27:07 {prompt: Перепиши сообщение клиента вежливо и без оскорблений, сохранив смысл и суть претензии:nnВаши ушлёпки-курьеры снова привезли мне разбитый заказ, а поддержка из тупых идиотов вообще отказалась помогать и просто послала нахуй, з

In [29]:
!pkill -f "nvidia-smi --query-gpu"; pkill vmstat
!awk '!seen[substr($0,21)]++' logs/train.log | tail -40

^C
2026-09-29 06:27:05 {prompt: Перепиши сообщение клиента вежливо и без оскорблений, сохранив смысл и суть претензии:nnВаши ушлёпки-курьеры снова привезли мне разбитый заказ, а поддержка из тупых идиотов вообще отказалась помогать и просто послала нахуй, заебали уже работать как дерьмо., chosen: Ваши безответственные курьеры снова доставили мне повреждённый товар, а сотрудники поддержки отказались оформить возврат и не предложили альтернативных решений, что вызывает сильное недовольство качеством вашего сервиса., rejected: Ваши безответственные курьеры снова доставили мне повреждённые товар, а сотрудники поддержки отказались оформить возврат и не предложили альтернативных решений, что вызывает сильное недовольство качеством ваш сервис.}



In [30]:
!tail -15 logs/vmstat_train.log

 1  0      0 1156848 210044 6496604    0    0     0   476 1435 1537 23  6 70  1  0  0 2026-09-29 06:34:57
 0  0      0 1157296 210044 6496632    0    0     0   432 1390 1858  9 10 80  1  0  0 2026-09-29 06:34:59
 0  0      0 1157044 210044 6496632    0    0     0   432  457  601  1  1 97  1  0  0 2026-09-29 06:35:01
 0  0      0 1477392 210052 6496608    0    0     0   540 1107 1557 12  3 84  1  0  0 2026-09-29 06:35:03
 0  0      0 1476664 210052 6496636    0    0     0   424  435  602  0  1 98  1  0  0 2026-09-29 06:35:05
 0  0      0 1476664 210052 6496636    0    0     0   554  426  600  1  1 98  1  0  0 2026-09-29 06:35:07
 3  0      0 1477556 210060 6496752    0    0   178   500 1924 2080 30  9 61  1  0  0 2026-09-29 06:35:09
 0  0      0 1475184 210060 6496688    0    0     0   430  880 1324  3  6 90  1  0  0 2026-09-29 06:35:11
 0  0      0 1473168 210060 6496676    0    0     0   432  468  617  1  1 97  1  0  0 2026-09-29 06:35:13
 1  0      0 1485264 210072 6497964    0    0 

In [31]:
!ps aux --sort=-%cpu | head -8

USER         PID %CPU %MEM    VSZ   RSS TTY      STAT START   TIME COMMAND
root        4684  6.6 25.0 4809804 3333436 ?     Sl   06:04   2:05 node /datalab/web/pyright/pyright-langserver.js --stdio --cancellationReceive=file:15d2b2d635e9c16d3c06f134bdac293fca5d84a3f6
root        3804  1.9 13.9 14253272 1851284 ?    Ssl  06:01   0:39 /usr/bin/python3 -m colab_kernel_launcher -f /root/.local/share/jupyter/runtime/kernel-2c2f9375-1cb6-4e8b-8095-6072b4a4077e.json
root          65  0.5  0.0      0     0 ?        Z    05:46   0:14 [python3] <defunct>
root         124  0.2  1.2 555748 164548 ?       Sl   05:46   0:07 /usr/bin/python3 /usr/local/bin/jupyter-server --debug --transport="ipc" --ip=172.28.0.12 --ServerApp.token= --port=9000 --FileContentsManager.root_dir=/ --FileContentsManager.allow_hidden=True --ServerApp.log_format="|%(levelname)s|%(message)s" --ServerApp.iopub_data_rate_limit=1e10 --MappingKernelManager.root_dir=/content
root           7  0.0  0.4 1296012 64304 ?       Sl   05

In [32]:
!mv logs/train.log logs/train_run1_hung_on_prompt.log
!mv logs/nvidia_smi_train.csv logs/nvidia_smi_run1_hung.csv
!mv logs/vmstat_train.log logs/vmstat_run1_hung.log

In [34]:
!nohup nvidia-smi --query-gpu=timestamp,memory.used,memory.total,utilization.gpu,power.draw --format=csv -l 1 > logs/nvidia_smi_train.csv 2>&1 &
!nohup vmstat -t 2 > logs/vmstat_train.log 2>&1 &
!sleep 3 && head -3 logs/nvidia_smi_train.csv

timestamp, memory.used [MiB], memory.total [MiB], utilization.gpu [%], power.draw [W]
2026/09/29 06:38:15.732, 107 MiB, 15360 MiB, 0 %, 33.94 W
2026/09/29 06:38:16.738, 107 MiB, 15360 MiB, 0 %, 33.94 W


In [35]:
!soup train --config config.yaml --yes --name dpo_run1 2>&1 | while IFS= read -r l; do echo "$(date '+%F %T') $l"; done | tee logs/train.log

2026-09-29 06:38:35 {prompt: Перепиши сообщение клиента вежливо и без оскорблений, сохранив смысл и суть претензии:nnВаши ушлёпки-курьеры снова привезли мне разбитый заказ, а поддержка из тупых идиотов вообще отказалась помогать и просто послала нахуй, заебали уже работать как дерьмо., chosen: Ваши безответственные курьеры снова доставили мне повреждённый товар, а сотрудники поддержки отказались оформить возврат и не предложили альтернативных решений, что вызывает сильное недовольство качеством вашего сервиса., rejected: Ваши безответственные курьеры снова доставили мне повреждённые товар, а сотрудники поддержки отказались оформить возврат и не предложили альтернативных решений, что вызывает сильное недовольство качеством ваш сервис.}

2026-09-29 06:38:37 {prompt: Перепиши сообщение клиента вежливо и без оскорблений, сохранив смысл и суть претензии:nnВаши ушлёпки-курьеры снова привезли мне разбитый заказ, а поддержка из тупых идиотов вообще отказалась помогать и просто послала нахуй, з

In [37]:
!pkill -f "nvidia-smi --query-gpu"; pkill vmstat
!mv logs/train.log logs/train_run2_garbled_by_wrapper.log
!mv logs/nvidia_smi_train.csv logs/nvidia_smi_run2.csv
!mv logs/vmstat_train.log logs/vmstat_run2.log

^C


In [38]:
!ls -la output/ 2>&1 | head -20
!soup runs list 2>&1 | tail -15

ls: cannot access 'output/': No such file or directory
Usage: soup runs [OPTIONS] [COMMAND] [ARGS]...
Try 'soup runs --help' for help.
╭─ Error ──────────────────────────────────────────────────────────────────────╮
│ No such command 'list'.                                                      │
╰──────────────────────────────────────────────────────────────────────────────╯


In [39]:
import pandas as pd
df = pd.read_csv("logs/nvidia_smi_run2.csv", skipinitialspace=True)
mem = pd.to_numeric(df["memory.used [MiB]"].astype(str).str.replace(" MiB", ""), errors="coerce")
print("rows:", len(df), "| peak GPU (nvidia-smi):", mem.max(), "MiB")
print(df.iloc[[0, len(df)//2, -1]])

rows: 129 | peak GPU (nvidia-smi): 247 MiB
                   timestamp memory.used [MiB] memory.total [MiB]  \
0    2026/09/29 06:38:15.732           107 MiB          15360 MiB   
64   2026/09/29 06:39:20.143           107 MiB          15360 MiB   
128  2026/09/29 06:40:24.483           107 MiB          15360 MiB   

    utilization.gpu [%] power.draw [W]  
0                   0 %        33.94 W  
64                  0 %        34.04 W  
128                 0 %        33.94 W  


In [40]:
!nohup nvidia-smi --query-gpu=timestamp,memory.used,memory.total,utilization.gpu,power.draw --format=csv -l 1 > logs/nvidia_smi_train.csv 2>&1 &
!nohup vmstat -t 2 > logs/vmstat_train.log 2>&1 &

In [41]:
!soup train --config config.yaml --yes --name dpo_run3 2>&1 | awk '{ print strftime("%Y-%m-%d %H:%M:%S"), $0; fflush() }' | tee logs/train.log

2026-09-29 06:43:05 Loading config from config.yaml...
2026-09-29 06:43:05 ╭─────────────────────────────── Training Setup ───────────────────────────────╮
2026-09-29 06:43:05 │ Device:  Tesla T4                                                            │
2026-09-29 06:43:05 │ Memory:  14.6 GB                                                             │
2026-09-29 06:43:05 │ Model:   Qwen/Qwen2.5-0.5B-Instruct                                          │
2026-09-29 06:43:05 │ Task:    dpo                                                                 │
2026-09-29 06:43:05 │ Backend: transformers                                                        │
2026-09-29 06:43:05 │ LoRA:    r=16, alpha=32                                                      │
2026-09-29 06:43:05 │ Quant:   4bit                                                                │
2026-09-29 06:43:05 ╰──────────────────────────────────────────────────────────────────────────────╯
2026-09-29 06:43:05 Loading dataset.

In [42]:
from transformers import AutoTokenizer
import json, numpy as np
tok = AutoTokenizer.from_pretrained("Qwen/Qwen2.5-0.5B-Instruct")
rows = [json.loads(x) for x in open("data/train.jsonl", encoding="utf-8")]
def n(p, c): return len(tok.apply_chat_template([{"role":"user","content":p},{"role":"assistant","content":c}], tokenize=True))
lc = np.array([n(r["prompt"], r["chosen"]) for r in rows]); lr = np.array([n(r["prompt"], r["rejected"]) for r in rows])
for name, a in [("prompt+chosen", lc), ("prompt+rejected", lr)]:
    print(f"{name:16s} p50={np.median(a):.0f} p95={np.percentile(a,95):.0f} max={a.max()}")
print("> 512 tokens:", int((np.maximum(lc, lr) > 512).sum()))

prompt+chosen    p50=208 p95=311 max=423
prompt+rejected  p50=218 p95=329 max=431
> 512 tokens: 0


In [43]:
%%writefile config.yaml
base: Qwen/Qwen2.5-0.5B-Instruct
task: dpo
data:
  train: ./data/train.jsonl
  format: dpo
  max_length: 512
  val_split: 0.1
training:
  epochs: 2
  lr: 1e-5
  batch_size: 4
  quantization: 4bit
  stream_layers: true
  stream_source: ram
  stream_buffers: 2
  gradient_checkpointing: true
  lora:
    r: 16
    alpha: 32
output: ./output

Overwriting config.yaml


In [44]:
!pkill -f "nvidia-smi --query-gpu"; pkill vmstat
!mv logs/train.log logs/train_run3_preflight_36GB.log
!mv logs/nvidia_smi_train.csv logs/nvidia_smi_run3.csv; mv logs/vmstat_train.log logs/vmstat_run3.log
!nohup nvidia-smi --query-gpu=timestamp,memory.used,memory.total,utilization.gpu,power.draw --format=csv -l 1 > logs/nvidia_smi_train.csv 2>&1 &
!nohup vmstat -t 2 > logs/vmstat_train.log 2>&1 &

^C


In [45]:
!soup train --config config.yaml --yes --name dpo_run4 2>&1 | awk '{ print strftime("%Y-%m-%d %H:%M:%S"), $0; fflush() }' | tee logs/train.log

2026-09-29 06:46:06 Loading config from config.yaml...
2026-09-29 06:46:06 ╭─────────────────────────────── Training Setup ───────────────────────────────╮
2026-09-29 06:46:06 │ Device:  Tesla T4                                                            │
2026-09-29 06:46:06 │ Memory:  14.6 GB                                                             │
2026-09-29 06:46:06 │ Model:   Qwen/Qwen2.5-0.5B-Instruct                                          │
2026-09-29 06:46:06 │ Task:    dpo                                                                 │
2026-09-29 06:46:06 │ Backend: transformers                                                        │
2026-09-29 06:46:06 │ LoRA:    r=16, alpha=32                                                      │
2026-09-29 06:46:06 │ Quant:   4bit                                                                │
2026-09-29 06:46:06 ╰──────────────────────────────────────────────────────────────────────────────╯
2026-09-29 06:46:06 Loading dataset.

In [46]:
!pkill -f "nvidia-smi --query-gpu"; pkill vmstat
!mv logs/train.log logs/train_run4_meta_crash.log
!mv logs/nvidia_smi_train.csv logs/nvidia_smi_run4.csv; mv logs/vmstat_train.log logs/vmstat_run4.log
!mkdir -p logs/crash && cp /content/.soup-crashes/*.crash logs/crash/
!file logs/crash/*.crash

^C
logs/crash/crash_20260929T064603Z_78106ccf.crash: JSON text data


In [47]:
!PYTHONUNBUFFERED=1 soup --verbose train --config config.yaml --yes --name dpo_run4_verbose 2>&1 | awk '{ print strftime("%Y-%m-%d %H:%M:%S"), $0; fflush() }' | tee logs/train_run4_verbose.log | tail -45

2026-09-29 06:48:53 │     return fn(*args, **kwargs)                                               │
2026-09-29 06:48:53 │   File                                                                       │
2026-09-29 06:48:53 │ "/usr/local/lib/python3.13/dist-packages/torch/_prims_common/wrappers.py",   │
2026-09-29 06:48:53 │ line 152, in _fn                                                             │
2026-09-29 06:48:53 │     result = fn(**bound.arguments)                                           │
2026-09-29 06:48:53 │   File "/usr/local/lib/python3.13/dist-packages/torch/_refs/__init__.py",    │
2026-09-29 06:48:53 │ line 1171, in _ref                                                           │
2026-09-29 06:48:53 │     output = prim(a, b)                                                      │
2026-09-29 06:48:53 │   File "/usr/local/lib/python3.13/dist-packages/torch/_refs/__init__.py",    │
2026-09-29 06:48:53 │ line 1790, in mul                                                    

In [48]:
!grep -nE "soup|peft|transformers/models|trl/" logs/train_run4_verbose.log | grep -v "Loading\|Run ID" | head -30

17:2026-09-29 06:48:53 Preparing layer shards -> /root/.soup/layer-stream/Qwen__Qwen2.5-0.5B-Instruct
18:2026-09-29 06:48:53 ╭───────────────────────── soup train --stream-layers ─────────────────────────╮
53:2026-09-29 06:48:53 /content/.soup-crashes/crash_20260929T064850Z_7bc61258.crash
61:2026-09-29 06:48:53 │   File "/usr/local/lib/python3.13/dist-packages/soup_cli/cli.py", line 810,  │
100:2026-09-29 06:48:53 │   File "/usr/local/lib/python3.13/dist-packages/soup_cli/commands/train.py", │
106:2026-09-29 06:48:53 │   File "/usr/local/lib/python3.13/dist-packages/soup_cli/trainer/dpo.py",    │
175:2026-09-29 06:48:53 │ "/usr/local/lib/python3.13/dist-packages/transformers/models/qwen2/modeling_ │
186:2026-09-29 06:48:53 │ "/usr/local/lib/python3.13/dist-packages/transformers/models/qwen2/modeling_ │


In [49]:
!grep gradient config.yaml
!nohup nvidia-smi --query-gpu=timestamp,memory.used,memory.total,utilization.gpu,power.draw --format=csv -l 1 > logs/nvidia_smi_train.csv 2>&1 &
!nohup vmstat -t 2 > logs/vmstat_train.log 2>&1 &
!PYTHONUNBUFFERED=1 soup train --config config.yaml --yes --name dpo_run5 2>&1 | awk '{ print strftime("%Y-%m-%d %H:%M:%S"), $0; fflush() }' | tee logs/train.log | tail -40

  gradient_checkpointing: true
2026-09-29 06:51:03 │ Layer streaming BETA — arch qwen2, tier ram                                  │
2026-09-29 06:51:03 │   base store   0.18 GB across 24 layers (pinned)                             │
2026-09-29 06:51:03 │   VRAM buffers 2 x 8 MB = 15 MB                                              │
2026-09-29 06:51:03 │   resident     272 MB embeddings + adapters                                  │
2026-09-29 06:51:03 │   peak VRAM    ~9.33 GB at batch 4 x seq 512 (8 rows — chosen+rejected are   │
2026-09-29 06:51:03 │ one concatenated tensor) (logits 8.71 GB)                                    │
2026-09-29 06:51:03 │   free VRAM    15.42 GB                                                      │
2026-09-29 06:51:03 │   forecast     378-556 tok/s — a compute-bound bound, not a promise          │
2026-09-29 06:51:03 │                (from 1.65 TFLOPS measured on this card now @ 630 MHz)        │
2026-09-29 06:51:03 │   ! accumulating 4x at batch 4: the ba

In [50]:
!sed -i 's/gradient_checkpointing: true/gradient_checkpointing: false/' config.yaml
!grep gradient config.yaml

  gradient_checkpointing: false


In [51]:
!pkill -f "nvidia-smi --query-gpu"; pkill vmstat
!nohup nvidia-smi --query-gpu=timestamp,memory.used,memory.total,utilization.gpu,power.draw --format=csv -l 1 > logs/nvidia_smi_train.csv 2>&1 &
!nohup vmstat -t 2 > logs/vmstat_train.log 2>&1 &
!PYTHONUNBUFFERED=1 soup train --config config.yaml --yes --name dpo_run5 2>&1 | awk '{ print strftime("%Y-%m-%d %H:%M:%S"), $0; fflush() }' | tee logs/train.log | tail -40

^C
2026-09-29 06:52:44 │ Layer streaming BETA — arch qwen2, tier ram                                  │
2026-09-29 06:52:44 │   base store   0.18 GB across 24 layers (pinned)                             │
2026-09-29 06:52:44 │   VRAM buffers 2 x 8 MB = 15 MB                                              │
2026-09-29 06:52:44 │   resident     272 MB embeddings + adapters                                  │
2026-09-29 06:52:44 │   peak VRAM    ~9.33 GB at batch 4 x seq 512 (8 rows — chosen+rejected are   │
2026-09-29 06:52:44 │ one concatenated tensor) (logits 8.71 GB)                                    │
2026-09-29 06:52:44 │   free VRAM    15.42 GB                                                      │
2026-09-29 06:52:44 │   forecast     374-550 tok/s — a compute-bound bound, not a promise          │
2026-09-29 06:52:44 │                (from 1.63 TFLOPS measured on this card now @ 870 MHz)        │
2026-09-29 06:52:44 │   ! accumulating 4x at batch 4: the base is re-read once per micro

In [52]:
!pkill -f "nvidia-smi --query-gpu"; pkill vmstat
!mv logs/train.log logs/train_run5_gc_off_meta_crash.log
!mv logs/nvidia_smi_train.csv logs/nvidia_smi_run5.csv; mv logs/vmstat_train.log logs/vmstat_run5.log
!mkdir -p logs/crash && cp /content/.soup-crashes/*.crash logs/crash/ 2>/dev/null; ls logs/crash
!sed -i 's/gradient_checkpointing: false/gradient_checkpointing: true/' config.yaml && grep gradient config.yaml

^C
crash_20260929T064603Z_78106ccf.crash  crash_20260929T065100Z_8fd22c68.crash
crash_20260929T064850Z_7bc61258.crash  crash_20260929T065241Z_d865e779.crash
  gradient_checkpointing: true


## Plan A summary: soup-cli 0.72.4 (as installed by the instructions)

**Environment:** Colab T4, Python 3.13.15, torch 2.11.0+cu128, transformers 4.57.6, peft 0.20.0, trl 0.24.0.

**Version:** `pip install "soup-cli[train]"` silently installs **0.72.4**, because every version ≥ 0.73.0 requires Python < 3.13 (`logs/pip_why_0724.txt`). Fixes missed as a result: 0.73.0 (wrong NF4 gradients), 0.73.1 (bf16 on pre-Ampere/T4), 0.73.2 (`soup ship` scoring), 0.74.0 (base loaded in fp32).

| Run | What I changed | Result | Log |
|---|---|---|---|
| 1 | base config | hung for 8+ min: GPU at 0.1 GB, CPU idle. Soup was waiting for a confirmation, and my log wrapper hid the prompt (`read` waits for `\n`) | `train_run1_hung_on_prompt.log` |
| 2 | `--yes` | failed after ~2 min, but all output was replaced: IPython substituted the Python variable `l` for `$l` in the wrapper | `train_run2_garbled_by_wrapper.log` |
| 3 | wrapper switched to `awk` | preflight: 36.28 GB needed, 15.42 GB free (default `max_length=2048`) | `train_run3_preflight_36GB.log` |
| 4 | `max_length: 512` | preflight 9.33 GB → OK; crash at step 0: `Tensor on device cuda:0 is not on the expected device meta!` + warning `None of the inputs have requires_grad=True` | `train_run4_meta_crash.log`, `train_run4_verbose.log` |
| 5 | `gradient_checkpointing: false` | same crash, warning still there → double-checkpointing hypothesis rejected | `train_run5_gc_off_meta_crash.log` |

**Memory estimates for the same config:** my estimate 2.1–5.3 GB · `soup profile` 2.0 GB · `soup plan` 8.0 GB · streaming preflight 36.28 GB (seq 2048) → 9.33 GB (seq 512, of which logits 8.71 GB). Actual peak not measured: no run reached step 1.

**Other findings:**
- Soup silently holds out 10% of train for validation (450 → 405).
- LoRA targets only `q_proj`/`v_proj` by default: 1,081,344 parameters (`profile` reported 4.82M).
- Hidden 4× gradient accumulation (effective batch 16, 52 steps).
- `torch.cuda.is_bf16_supported()` returns True on a T4, even though it has no hardware bf16.
- `data doctor` refuses DPO data; `lint` reports `near_duplicates` as "OK" even though the check was skipped; empty `rejected` values are caught by nothing.

**Plan A conclusion:** on the version the instructions install, DPO + layer streaming does not train on Colab at all. The failure itself is loud, but its root cause (the silent resolution to 0.72.4) is silent.

**AI tools:** the assistant (Claude) proposed the log wrapper used in runs 1–2. I found both bugs myself (the hidden prompt and the replaced output): from an empty GPU with an idle CPU, and from identical log lines.


In [53]:
!pip install -q uv
!uv venv -p 3.12 /content/py312
!uv pip install -p /content/py312/bin/python "soup-cli[train]==0.75.1" 2>&1 | tail -5

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 20.5/20.5 MB 81.9 MB/s eta 0:00:00
Using CPython 3.12.3 interpreter at: /usr/bin/python3.12
Creating virtual environment at: py312
Activate with: source py312/bin/activate
 + typing-extensions==4.16.0
 + typing-inspection==0.4.4
 + urllib3==2.8.0
 + xxhash==4.0.1
 + yarl==1.25.1


In [54]:
!/content/py312/bin/soup version
!/content/py312/bin/python -c "import torch, transformers, peft, trl; print('torch', torch.__version__, '| tf', transformers.__version__, '| peft', peft.__version__, '| trl', trl.__version__, '| cuda', torch.cuda.is_available())" | tee logs/env_py312.txt

soup v0.75.1
https://github.com/MakazhanAlpamys/Soup
torch 2.14.0+cu130 | tf 5.17.0 | peft 0.21.0 | trl 0.29.1 | cuda True


In [55]:
!mv /root/.soup/layer-stream /root/.soup/layer-stream_v0724_backup

In [56]:
!nohup nvidia-smi --query-gpu=timestamp,memory.used,memory.total,utilization.gpu,power.draw --format=csv -l 1 > logs/nvidia_smi_train.csv 2>&1 &
!nohup vmstat -t 2 > logs/vmstat_train.log 2>&1 &
!PYTHONUNBUFFERED=1 /content/py312/bin/soup train --config config.yaml --yes --name dpo_run6_v0751 2>&1 | awk '{ print strftime("%Y-%m-%d %H:%M:%S"), $0; fflush() }' | tee logs/train.log | tail -50

2026-09-29 07:01:23 Loading config from config.yaml...
2026-09-29 07:01:23 ╭─────────────────────────────── Training Setup ───────────────────────────────╮
2026-09-29 07:01:23 │ Device:  Tesla T4                                                            │
2026-09-29 07:01:23 │ Memory:  14.6 GB                                                             │
2026-09-29 07:01:23 │ Model:   Qwen/Qwen2.5-0.5B-Instruct                                          │
2026-09-29 07:01:23 │ Task:    dpo                                                                 │
2026-09-29 07:01:23 │ Backend: transformers                                                        │
2026-09-29 07:01:23 │ LoRA:    r=16, alpha=32                                                      │
2026-09-29 07:01:23 │ Quant:   4bit                                                                │
2026-09-29 07:01:23 │ Seed:    42 (unset default)                                                  │
2026-09-29 07:01:23 ╰───────────────

In [57]:
!pkill -f "nvidia-smi --query-gpu"; pkill vmstat
!mv logs/train.log logs/train_run6_v0751_hf_cache_error.log
!mv logs/nvidia_smi_train.csv logs/nvidia_smi_run6.csv; mv logs/vmstat_train.log logs/vmstat_run6.log

^C


In [58]:
!nohup nvidia-smi --query-gpu=timestamp,memory.used,memory.total,utilization.gpu,power.draw --format=csv -l 1 > logs/nvidia_smi_train.csv 2>&1 &
!nohup vmstat -t 2 > logs/vmstat_train.log 2>&1 &
!HF_HOME=/content/hf_cache_312 PYTHONUNBUFFERED=1 /content/py312/bin/soup train --config config.yaml --yes --name dpo_run7_v0751 2>&1 | awk '{ print strftime("%Y-%m-%d %H:%M:%S"), $0; fflush() }' | tee logs/train.log | tail -50

2026-09-29 07:03:33 Loading config from config.yaml...
2026-09-29 07:03:33 ╭─────────────────────────────── Training Setup ───────────────────────────────╮
2026-09-29 07:03:33 │ Device:  Tesla T4                                                            │
2026-09-29 07:03:33 │ Memory:  14.6 GB                                                             │
2026-09-29 07:03:33 │ Model:   Qwen/Qwen2.5-0.5B-Instruct                                          │
2026-09-29 07:03:33 │ Task:    dpo                                                                 │
2026-09-29 07:03:33 │ Backend: transformers                                                        │
2026-09-29 07:03:33 │ LoRA:    r=16, alpha=32                                                      │
2026-09-29 07:03:33 │ Quant:   4bit                                                                │
2026-09-29 07:03:33 │ Seed:    42 (unset default)                                                  │
2026-09-29 07:03:33 ╰───────────────

In [59]:
!find /content/hf_cache_312/hub -name "model.safetensors" -exec ls -la {} \; | tee logs/run7_hf_cache_diag.txt
!/content/py312/bin/python -c "import huggingface_hub; print('huggingface_hub', huggingface_hub.__version__)" | tee -a logs/run7_hf_cache_diag.txt


lrwxrwxrwx 1 root root 76 Sep 29 07:03 /content/hf_cache_312/hub/models--Qwen--Qwen2.5-0.5B-Instruct/snapshots/7ae557604adf67be50417f59c2c2f167def9a775/model.safetensors -> ../../blobs/fdf756fa7fcbe7404d5c60e26bff1a0c8b8aa1f72ced49e7dd0210fe288fb7fe
huggingface_hub 1.33.0


In [60]:
!pkill -f "nvidia-smi --query-gpu"; pkill vmstat
!mv logs/train.log logs/train_run7_v0751_hf_cache_error_fresh.log
!mv logs/nvidia_smi_train.csv logs/nvidia_smi_run7.csv; mv logs/vmstat_train.log logs/vmstat_run7.log
!/content/py312/bin/python -c "from huggingface_hub import snapshot_download; print(snapshot_download('Qwen/Qwen2.5-0.5B-Instruct', local_dir='/content/models/qwen2.5-0.5b-instruct'))"
!ls -la /content/models/qwen2.5-0.5b-instruct
!sed -i 's#^base: .*#base: /content/models/qwen2.5-0.5b-instruct#' config.yaml && head -3 config.yaml

^C

/content/models/qwen2.5-0.5b-instruct


total 976212
drwxr-xr-x 3 root root      4096 Sep 29 07:05 .
drwxr-xr-x 3 root root      4096 Sep 29 07:05 ..
drwxr-xr-x 3 root root      4096 Sep 29 07:05 .cache
-rw-r--r-- 1 root root       659 Sep 29 07:05 config.json
-rw-r--r-- 1 root root       242 Sep 29 07:05 generation_config.json
-rw-r--r-- 1 root root      1519 Sep 29 07:05 .gitattributes
-rw-r--r-- 1 root root     11343 Sep 29 07:05 LICENSE
-rw-r--r-- 1 root root   1671839 Sep 29 07:05 merges.txt
-rw-r--r-- 1 root root 988097824 Sep 29 07:05 model.safetensors
-rw-r--r-- 1 root root      4917 Sep 29 07:05 README.md
-rw-r--r-- 1 root root      7305 Sep 29 07:05 tokenizer_config.json
-rw-r--r-- 1 root root   7031645 Sep 29 07:05 tokenizer.json
-rw-r--r-- 1 root root   2776833 Sep 29 07:05 vocab.json
base: /content/models/qwen2.5-0.5b-instruct
task: dpo
data:


In [61]:
!nohup nvidia-smi --query-gpu=timestamp,memory.used,memory.total,utilization.gpu,power.draw --format=csv -l 1 > logs/nvidia_smi_train.csv 2>&1 &
!nohup vmstat -t 2 > logs/vmstat_train.log 2>&1 &
!PYTHONUNBUFFERED=1 /content/py312/bin/soup train --config config.yaml --yes --name dpo_run8_v0751_localbase 2>&1 | awk '{ print strftime("%Y-%m-%d %H:%M:%S"), $0; fflush() }' | tee logs/train.log | tail -50

2026-09-29 07:09:16 │ VRAM flat, so raise batch_size while the budget above allows, then           │
2026-09-29 07:09:16 │ accumulate for the rest.                                                     │
2026-09-29 07:09:16 ╰──────────────────────────────────────────────────────────────────────────────╯
2026-09-29 07:09:16 Layer streaming is BETA: slower than resident training, but this model may not 
2026-09-29 07:09:16 run resident on this card at all.
2026-09-29 07:09:16 expandable_segments allocator hint not enabled: CUDA was already initialised 
2026-09-29 07:09:16 before Soup could set it — the allocator reads PYTORCH_CUDA_ALLOC_CONF once, at 
2026-09-29 07:09:16 context creation
2026-09-29 07:09:16 Layer streaming ready: 24 layers, 0.18 GB pinned RAM store, 2 x 8 MB decoder 
2026-09-29 07:09:16 buffers + 1 x 0 MB large-layer slot
2026-09-29 07:09:16 LoRA applied: 1,081,344 trainable / 494,032,768 total (0.22%)
Map: 100%|██████████| 45/45 [00:00<00:00, 4404.75 examples/s]
2026-09-2

In [72]:
%%writefile verify.py
"""verify.py: проверяет, что DPO-адаптер реально обучился и изменил модель.
  python verify.py --base <model_dir> --adapter <adapter_dir> --out logs/verify_run.json
  python verify.py --base <model_dir> --control-fresh --out logs/verify_control.json   # контроль: должен быть NOT PROVEN
"""
import argparse, json, random, statistics, time
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import PeftModel, LoraConfig, get_peft_model

ap = argparse.ArgumentParser()
ap.add_argument("--base", required=True)
ap.add_argument("--adapter")
ap.add_argument("--control-fresh", action="store_true", help="свежий НЕобученный LoRA (B=0) как отрицательный контроль")
ap.add_argument("--eval", default="data/eval.jsonl")
ap.add_argument("--meta", default="data/eval_meta.jsonl")
ap.add_argument("--beta", type=float, default=0.1)
ap.add_argument("--fp16-base", action="store_true", help="база в fp16 вместо NF4")
ap.add_argument("--out", default="logs/verify.json")
a = ap.parse_args()
torch.manual_seed(0); random.seed(0); t0 = time.time()

tok = AutoTokenizer.from_pretrained(a.base)
q = None if a.fp16_base else BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4", bnb_4bit_compute_dtype=torch.float16)
base = AutoModelForCausalLM.from_pretrained(a.base, torch_dtype=torch.float16, quantization_config=q, device_map={"": 0})
if a.control_fresh:
    model = get_peft_model(base, LoraConfig(r=16, lora_alpha=32, target_modules=["q_proj", "v_proj"], task_type="CAUSAL_LM"))
    name = "CONTROL: fresh untrained LoRA"
else:
    model = PeftModel.from_pretrained(base, a.adapter); name = a.adapter
model.eval()

# [1] Веса адаптера: lora_B инициализируется нулями -> после обучения должна стать ненулевой
bn = [p.detach().float().norm().item() for n, p in model.named_parameters() if "lora_B" in n]
nz = sum(v > 1e-8 for v in bn)

EOT = "<|im_end|>\n"
@torch.no_grad()
def logp(prompt, completion):
    ptxt = tok.apply_chat_template([{"role": "user", "content": prompt}], tokenize=False, add_generation_prompt=True)
    p_ids = tok(ptxt, add_special_tokens=False).input_ids
    c_ids = tok(completion + EOT, add_special_tokens=False).input_ids
    ids = torch.tensor([p_ids + c_ids], device="cuda")
    lp = torch.log_softmax(model(ids).logits[0, :-1].float(), -1).gather(-1, ids[0, 1:, None]).squeeze(-1)
    return lp[len(p_ids) - 1:].sum().item(), len(c_ids)

data = [json.loads(l) for l in open(a.eval, encoding="utf-8")]
meta = [json.loads(l) for l in open(a.meta, encoding="utf-8")] if a.meta else [{}] * len(data)
res = []
for ex, m in zip(data, meta):
    ct, lc = logp(ex["prompt"], ex["chosen"]); rt, lr = logp(ex["prompt"], ex["rejected"])
    with model.disable_adapter():
        cb, _ = logp(ex["prompt"], ex["chosen"]); rb, _ = logp(ex["prompt"], ex["rejected"])
    res.append(dict(type=m.get("rejected_type", "?"), ct=ct, rt=rt, cb=cb, rb=rb, lc=lc, lr=lr,
                    margin=a.beta * ((ct - cb) - (rt - rb)), shift=max(abs(ct - cb), abs(rt - rb))))

n = len(res)
acc_b = sum(r["cb"] > r["rb"] for r in res) / n
acc_t = sum(r["ct"] > r["rt"] for r in res) / n
margins = [r["margin"] for r in res]
max_shift = max(r["shift"] for r in res)
boots = sorted(statistics.fmean(random.choices(margins, k=n)) for _ in range(2000))
lo, hi = boots[50], boots[1949]
len_base = sum(r["lc"] < r["lr"] for r in res) / n          # "глупый" baseline: выбирать более короткий ответ
try: corr = statistics.correlation([r["lr"] - r["lc"] for r in res], margins)
except Exception: corr = float("nan")
by_type = {}
for r in res: by_type.setdefault(r["type"], []).append(r)

checks = {
    "1_adapter_B_nonzero":        len(bn) > 0 and nz == len(bn),
    "2_adapter_changes_logprobs": max_shift > 0.05,
    "3_heldout_margin_CI_above_0": lo > 0,
    "4_heldout_acc_improved":     acc_t > acc_b,
}
verdict = "REAL TRAINING" if all(checks.values()) else "NOT PROVEN"

print(f"=== verify.py | {name} | eval pairs: {n} (never seen in training) ===")
print(f"[1] lora_B tensors: {len(bn)}, nonzero: {nz}, mean norm: {statistics.fmean(bn) if bn else 0:.4e}")
print(f"[2] max |logp(adapter on) - logp(adapter off)|: {max_shift:.4f} nats")
print(f"[3] implicit reward margin (beta={a.beta}): mean {statistics.fmean(margins):+.4f}, 95% CI [{lo:+.4f}, {hi:+.4f}], positive {sum(x > 0 for x in margins)}/{n}")
print(f"[4] held-out preference accuracy: base {acc_b:.0%} -> adapter {acc_t:.0%}")
print(f"[5] length shortcut: baseline 'pick shorter' = {len_base:.0%}; corr(len_rej - len_cho, margin) = {corr:+.2f}")
print("[6] per rejected_type (n, acc base -> adapter, mean margin):")
for t, rs in sorted(by_type.items()):
    print(f"    {t:20s} n={len(rs):2d}  {sum(r['cb'] > r['rb'] for r in rs)/len(rs):.0%} -> {sum(r['ct'] > r['rt'] for r in rs)/len(rs):.0%}  {statistics.fmean(r['margin'] for r in rs):+.4f}")
for k, v in checks.items(): print(f"    {'PASS' if v else 'FAIL'}  {k}")
print(f"VERDICT: {verdict}   ({time.time() - t0:.0f}s)")
json.dump(dict(name=name, n=n, checks=checks, verdict=verdict, acc_base=acc_b, acc_adapter=acc_t,
               margin_mean=statistics.fmean(margins), margin_ci=[lo, hi], max_shift=max_shift,
               len_baseline=len_base, len_margin_corr=corr, rows=res), open(a.out, "w"), ensure_ascii=False, indent=1)

Overwriting verify.py


In [73]:
!pkill -f "nvidia-smi --query-gpu"; pkill vmstat
!cp logs/train.log logs/train_run8_v0751_success.log
!mv logs/nvidia_smi_train.csv logs/nvidia_smi_run8.csv; mv logs/vmstat_train.log logs/vmstat_run8.log
!ls -la output
!echo "eval_loss lines:" && grep -c "eval_loss" logs/train.log

^C
mv: cannot stat 'logs/nvidia_smi_train.csv': No such file or directory
mv: cannot stat 'logs/vmstat_train.log': No such file or directory
total 15436
drwxr-xr-x 4 root root     4096 Sep 29 07:09 .
drwxr-xr-x 1 root root     4096 Sep 29 07:14 ..
-rw-r--r-- 1 root root     1116 Sep 29 07:09 adapter_config.json
-rw------- 1 root root  4338000 Sep 29 07:09 adapter_model.safetensors
-rw-r--r-- 1 root root     2507 Sep 29 07:09 chat_template.jinja
drwxr-xr-x 3 root root     4096 Sep 29 07:09 checkpoint-52
-rw-r--r-- 1 root root     2373 Sep 29 07:09 README.md
drwxr-xr-x 2 root root     4096 Sep 29 07:09 ref
-rw-r--r-- 1 root root      693 Sep 29 07:09 tokenizer_config.json
-rw-r--r-- 1 root root 11421892 Sep 29 07:09 tokenizer.json
-rw-r--r-- 1 root root     5777 Sep 29 07:09 training_args.bin
eval_loss lines:
0


In [74]:
import pandas as pd
df = pd.read_csv("logs/nvidia_smi_run8.csv", skipinitialspace=True)
mem = pd.to_numeric(df["memory.used [MiB]"].astype(str).str.replace(" MiB", ""), errors="coerce")
util = pd.to_numeric(df["utilization.gpu [%]"].astype(str).str.replace(" %", ""), errors="coerce")
print(f"peak nvidia-smi: {mem.max()} MiB = {mem.max()/1024:.2f} GiB at {df.loc[mem.idxmax(), 'timestamp']}")
print(f"mean GPU util while training: {util[mem > 1024].mean():.0f}%  | seconds on GPU: {(mem > 1024).sum()}")

peak nvidia-smi: 13275 MiB = 12.96 GiB at 2026/09/29 07:06:11.109
mean GPU util while training: 92%  | seconds on GPU: 190


In [75]:
!/content/py312/bin/python verify.py --base /content/models/qwen2.5-0.5b-instruct --adapter output --out logs/verify_run8.json 2>&1 | tee logs/verify_run8.txt | tail -2

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|██████████| 290/290 [00:01<00:00, 201.35it/s]
=== verify.py | output | eval pairs: 50 (never seen in training) ===
[1] lora_B tensors: 48, nonzero: 48, mean norm: 7.2653e-03
[2] max |logp(adapter on) - logp(adapter off)|: 2.3203 nats
[3] implicit reward margin (beta=0.1): mean +0.0535, 95% CI [+0.0374, +0.0687], positive 42/50
[4] held-out preference accuracy: base 64% -> adapter 64%
[5] length shortcut: baseline 'pick shorter' = 72%; corr(len_rej - len_cho, margin) = +0.41
[6] per rejected_type (n, acc base -> adapter, mean margin):
    exact_copy           n= 1  0% -> 0%  +0.0978
    grammar_broken       n= 4  100% -> 100%  +0.0193
    hallucination        n= 7  100% -> 100%  +0.1003
    incomplete_rewrite   n= 9  33% -> 33%  +0.0767
    meaning_loss         n= 3  33% -> 33%  -0.0078
    over_softening       n= 6  83% -> 83%  +0.0429
    register_mismatch    n= 2  100% -> 100%  +0.1895
    residual

In [76]:
!/content/py312/bin/python verify.py --base /content/models/qwen2.5-0.5b-instruct --control-fresh --out logs/verify_control_fresh.json 2>&1 | tee logs/verify_control_fresh.txt | tail -25

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|██████████| 290/290 [00:01<00:00, 270.41it/s]
=== verify.py | CONTROL: fresh untrained LoRA | eval pairs: 50 (never seen in training) ===
[1] lora_B tensors: 48, nonzero: 0, mean norm: 0.0000e+00
[2] max |logp(adapter on) - logp(adapter off)|: 0.0000 nats
[3] implicit reward margin (beta=0.1): mean +0.0000, 95% CI [+0.0000, +0.0000], positive 0/50
[4] held-out preference accuracy: base 64% -> adapter 64%
[5] length shortcut: baseline 'pick shorter' = 72%; corr(len_rej - len_cho, margin) = +nan
[6] per rejected_type (n, acc base -> adapter, mean margin):
    exact_copy           n= 1  0% -> 0%  +0.0000
    grammar_broken       n= 4  100% -> 100%  +0.0000
    hallucination        n= 7  100% -> 100%  +0.0000
    incomplete_rewrite   n= 9  33% -> 33%  +0.0000
    meaning_loss         n= 3  33% -> 33%  +0.0000
    over_softening       n= 6  83% -> 83%  +0.0000
    register_mismatch    n= 2  100% -> 100%  

In [78]:
!PYTHONUNBUFFERED=1 /content/py312/bin/soup ship --base /content/models/qwen2.5-0.5b-instruct --adapter output --noise-floor 3 --device cuda --output logs/ship_run8.json --emit-evidence logs/ship_run8_evidence.json 2>&1 | awk '{ print strftime("%Y-%m-%d %H:%M:%S"), $0; fflush() }' | tee logs/ship_run8.txt | tail -60

2026-09-29 07:29:16 Error: live run needs --task-eval <tasks.jsonl> for the leg-1 task win


In [79]:
import glob, subprocess
roots = [p for p in glob.glob("/content/py312/lib/python3.12/site-packages/soup*") if not p.endswith("-info")]
print(roots)
out = subprocess.run(["grep", "-rn", "-E", "task_eval|task-eval", *roots], capture_output=True, text=True).stdout
print("\n".join(out.splitlines()[:40]))

['/content/py312/lib/python3.12/site-packages/soup_cli']
/content/py312/lib/python3.12/site-packages/soup_cli/config/schema.py:3969:    task_eval: Optional[str] = Field(
/content/py312/lib/python3.12/site-packages/soup_cli/config/schema.py:4044:        description="soup ship verdict defaults (task_eval / task_mode / "
/content/py312/lib/python3.12/site-packages/soup_cli/commands/ship.py:5:    soup ship --base <m> --adapter <lora> --task-eval tasks.jsonl
/content/py312/lib/python3.12/site-packages/soup_cli/commands/ship.py:542:    task_eval: str,
/content/py312/lib/python3.12/site-packages/soup_cli/commands/ship.py:546:    tasks = load_eval_tasks(task_eval)
/content/py312/lib/python3.12/site-packages/soup_cli/commands/ship.py:548:        raise ValueError(f"task-eval file {task_eval!r} has no tasks")
/content/py312/lib/python3.12/site-packages/soup_cli/commands/ship.py:555:    task_eval: str, judge_model: str
/content/py312/lib/python3.12/site-packages/soup_cli/commands/ship.py:568:    t

In [81]:
import inspect, sys
sys.path.insert(0, "/content/py312/lib/python3.12/site-packages")
src = open("/content/py312/lib/python3.12/site-packages/soup_cli/commands/ship.py", encoding="utf-8").read().splitlines()
print("\n".join(f"{i+1}: {l}" for i, l in enumerate(src) if "import" in l and "load_eval_tasks" in l))
print("----- _leg1_metric / score_task -----")
print("\n".join(f"{i+1}: {l}" for i, l in enumerate(src[535:555], start=535)))
print("\n".join(f"{i+1}: {l}" for i, l in enumerate(src[785:815], start=785)))

544:     from soup_cli.eval.custom import load_eval_tasks, run_eval
564:     from soup_cli.eval.custom import load_eval_tasks
624:     from soup_cli.eval.custom import load_eval_tasks
800:         from soup_cli.eval.custom import load_eval_tasks, run_eval
----- _leg1_metric / score_task -----
536: 
537: def _leg1_metric(
538:     base_gen: Callable[[str], str],
539:     tuned_gen: Callable[[str], str],
540:     base_id: str,
541:     tuned_id: str,
542:     task_eval: str,
543: ) -> TaskWin:
544:     from soup_cli.eval.custom import load_eval_tasks, run_eval
545: 
546:     tasks = load_eval_tasks(task_eval)
547:     if not tasks:
548:         raise ValueError(f"task-eval file {task_eval!r} has no tasks")
549:     base_acc = run_eval(base_id, tasks, generate_fn=base_gen).accuracy
550:     tuned_acc = run_eval(tuned_id, tasks, generate_fn=tuned_gen).accuracy
551:     return build_task_win("metric", base_acc, tuned_acc)
552: 
553: 
554: def _build_judge_scorer(
555:     task_eval: str, ju

In [82]:
out = subprocess.run(["grep", "-rn", "-A40", "def load_eval_tasks", *roots], capture_output=True, text=True).stdout
print(out[:4000])

/content/py312/lib/python3.12/site-packages/soup_cli/eval/custom.py:75:def load_eval_tasks(path: "Path | str") -> list[EvalTask]:
/content/py312/lib/python3.12/site-packages/soup_cli/eval/custom.py-76-    """Load eval tasks from a JSONL file with schema validation.
/content/py312/lib/python3.12/site-packages/soup_cli/eval/custom.py-77-
/content/py312/lib/python3.12/site-packages/soup_cli/eval/custom.py-78-    Each line must be a JSON object with at least a 'prompt' field.
/content/py312/lib/python3.12/site-packages/soup_cli/eval/custom.py-79-    Optional: 'expected', 'category', 'scoring'.
/content/py312/lib/python3.12/site-packages/soup_cli/eval/custom.py-80-    """
/content/py312/lib/python3.12/site-packages/soup_cli/eval/custom.py-81-    path = Path(path)
/content/py312/lib/python3.12/site-packages/soup_cli/eval/custom.py-82-    if not path.exists():
/content/py312/lib/python3.12/site-packages/soup_cli/eval/custom.py-83-        raise FileNotFoundError(f"Eval tasks file not found: {p

In [83]:
src = open("/content/py312/lib/python3.12/site-packages/soup_cli/eval/custom.py", encoding="utf-8").read().splitlines()
for i, l in enumerate(src):
    if "scoring" in l or "def _score" in l or "def score" in l or "exact" in l or "contains" in l or "regex" in l:
        print(f"{i+1}: {l}")

16:     "exact", "contains", "regex", "semantic",
28:     scoring: str = "exact"
79:     Optional: 'expected', 'category', 'scoring'.
118:             scoring = row.get("scoring", "exact")
119:             if scoring not in VALID_SCORING:
121:                     f"Line {line_num}: invalid scoring '{scoring}', "
129:                 scoring=scoring,
132:                     if k not in ("prompt", "expected", "category", "scoring")
142: def score_exact(output: str, expected: str) -> bool:
147: def score_contains(output: str, expected: str) -> bool:
152: def score_regex(output: str, expected: str) -> bool:
153:     """Check if output matches expected regex pattern.
166: def score_semantic(output: str, expected: str) -> float:
285:     "exact": score_exact,
286:     "contains": score_contains,
287:     "regex": score_regex,
293: def score_task(task: EvalTask, output: str) -> EvalResult:
295:     if task.scoring == "semantic":
302:     if task.scoring == "tool_call_args_subset":
309:     s

In [84]:
print("\n".join(src[165:200]))   # как считается semantic

import json
ev   = [json.loads(l) for l in open("data/eval.jsonl", encoding="utf-8")]
meta = [json.loads(l) for l in open("data/eval_meta.jsonl", encoding="utf-8")]
NO_PROFANITY = r"(?i)^(?![\s\S]*(ху[йеёяи]|бля|пизд|еба|ёба|идиот|дебил|дерьм|тупы|тупо|ушл[её]п|урод|мразь|сволоч|кретин|придур))[\s\S]{20,}$"

def write(path, rows):
    with open(path, "w", encoding="utf-8") as f:
        for r in rows: f.write(json.dumps(r, ensure_ascii=False) + "\n")

write("data/task_eval_exact.jsonl",    [{"prompt": e["prompt"], "expected": e["chosen"], "category": m["rejected_type"]} for e, m in zip(ev, meta)])
write("data/task_eval_semantic.jsonl", [{"prompt": e["prompt"], "expected": e["chosen"], "category": m["rejected_type"], "scoring": "semantic"} for e, m in zip(ev, meta)])
write("data/task_eval_detox.jsonl",    [{"prompt": e["prompt"], "expected": NO_PROFANITY, "category": m["rejected_type"], "scoring": "regex"} for e, m in zip(ev, meta)])
!wc -l data/task_eval_*.jsonl

def score_semantic(output: str, expected: str) -> float:
    """Compute similarity via Jaccard token overlap (bag-of-words).

    This is a lexical metric, not embedding-based semantic similarity.
    Returns a float 0-1. Considered a match if similarity >= 0.5.
    """
    out_tokens = set(output.strip().lower().split())
    exp_tokens = set(expected.strip().lower().split())
    if not out_tokens or not exp_tokens:
        return 0.0
    intersection = out_tokens & exp_tokens
    union = out_tokens | exp_tokens
    return len(intersection) / len(union)


def _parse_tool_call(text: str) -> Optional[dict]:
    """Parse a tool-call JSON blob. Returns None on any failure."""
    if not isinstance(text, str):
        return None
    try:
        data = json.loads(text)
    except (json.JSONDecodeError, ValueError):
        return None
    if not isinstance(data, dict):
        return None
    return data


def _extract_function(call: dict) -> Optional[dict]:
    func = call.get("function")

In [85]:
!PYTHONUNBUFFERED=1 /content/py312/bin/soup ship --base /content/models/qwen2.5-0.5b-instruct --adapter output --task-eval data/task_eval_semantic.jsonl --noise-floor 3 --device cuda --output logs/ship_semantic.json --emit-evidence logs/ship_semantic_evidence.json 2>&1 | awk '{ print strftime("%Y-%m-%d %H:%M:%S"), $0; fflush() }' | tee logs/ship_semantic.txt | tail -60

In [86]:
!mv logs/ship_semantic.txt logs/ship_semantic_stopped_50min.txt
!tail -40 logs/ship_semantic_stopped_50min.txt
!ls -la logs/ship_semantic* 2>/dev/null

-rw-r--r-- 1 root root 0 Sep 29 07:34 logs/ship_semantic_stopped_50min.txt


In [87]:
!nohup env PYTHONUNBUFFERED=1 /content/py312/bin/soup --log-level verbose ship --base /content/models/qwen2.5-0.5b-instruct --adapter output --task-eval data/task_eval_detox.jsonl --device cuda --output logs/ship_detox.json > logs/ship_detox.txt 2>&1 &

In [88]:
!sleep 60; tail -5 logs/ship_detox.txt; nvidia-smi --query-gpu=memory.used,utilization.gpu --format=csv,noheader

Live eval: loading base/tuned at full precision (bfloat16); pass --config to 
reuse the training run's own quantization.
[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|██████████| 290/290 [00:00<00:00, 4507.48it/s]
2251 MiB, 24 %


In [92]:
!mv logs/ship_detox.txt logs/ship_detox_bf16_default_stopped.txt 2>/dev/null
!nohup env PYTHONUNBUFFERED=1 /content/py312/bin/soup --log-level verbose ship --config config.yaml --base /content/models/qwen2.5-0.5b-instruct --adapter output --task-eval data/task_eval_detox.jsonl --device cuda --output logs/ship_detox.json > logs/ship_detox.txt 2>&1 &

In [93]:
!sleep 60; tail -5 logs/ship_detox.txt; nvidia-smi --query-gpu=memory.used,utilization.gpu --format=csv,noheader

Live eval: loading base/tuned at 4bit (reused from --config 
training.quantization).
Loading weights: 100%|██████████| 290/290 [00:00<00:00, 535.50it/s]
3311 MiB, 86 %


In [94]:
%%writefile gen_samples.py
import json, time, torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import PeftModel
BASE = "/content/models/qwen2.5-0.5b-instruct"
tok = AutoTokenizer.from_pretrained(BASE)
m = AutoModelForCausalLM.from_pretrained(BASE, dtype=torch.float16, device_map={"": 0},
      quantization_config=BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4", bnb_4bit_compute_dtype=torch.float16))
m = PeftModel.from_pretrained(m, "output").eval()
ev = [json.loads(l) for l in open("data/eval.jsonl", encoding="utf-8")][:3]
out = []
for i, e in enumerate(ev):
    txt = tok.apply_chat_template([{"role": "user", "content": e["prompt"]}], tokenize=False, add_generation_prompt=True)
    ids = tok(txt, return_tensors="pt").to("cuda")
    for mode in ["base", "adapter"]:
        t = time.time()
        with torch.no_grad():
            if mode == "base":
                with m.disable_adapter(): g = m.generate(**ids, max_new_tokens=256, do_sample=False)
            else: g = m.generate(**ids, max_new_tokens=256, do_sample=False)
        new = g[0, ids.input_ids.shape[1]:]
        ans = tok.decode(new, skip_special_tokens=True)
        stopped = tok.eos_token_id in new.tolist() or tok.convert_tokens_to_ids("<|im_end|>") in new.tolist()
        print(f"\n[{i}] {mode:7s} tokens={len(new):3d} stopped_on_eos={stopped} time={time.time()-t:.1f}s\n{ans}")
        out.append(dict(i=i, mode=mode, tokens=len(new), stopped=stopped, text=ans))
    print(f"\n[{i}] REFERENCE chosen:\n{e['chosen']}\n" + "-" * 80)
json.dump(out, open("logs/gen_samples.json", "w"), ensure_ascii=False, indent=1)

Overwriting gen_samples.py


In [95]:
!/content/py312/bin/python gen_samples.py 2>&1 | grep -v Warning | tee logs/gen_samples.txt

Loading weights: 100%|██████████| 290/290 [00:02<00:00, 128.36it/s]

[0] base    tokens= 78 stopped_on_eos=True time=13.8s
Вот перепись клиента вежливо и без оскорблений, сохраняя смысл и суть претензии:

"Ничего нового, это у нас все время. Мы просто хотим помочь вам с этим. Не пытайтесь обидеть нас или нам. Спасибо за вашу помощь!"

[0] adapter tokens= 78 stopped_on_eos=True time=11.9s
Вот перепись клиента вежливо и без оскорблений, сохраняя смысл и суть претензии:

"Ничего нового, это у нас все время. Мы просто хотим помочь вам с этим. Не пытайтесь обидеть нас или нам. Спасибо за вашу помощь!"

[0] REFERENCE chosen:
Опять этот представитель министерства анонсировал повышение тарифов на ЖКХ. Нам совершенно не подходят чиновники, которые не представляют реальных финансовых трудностей семей и фокусируются лишь на отчётных цифрах. Это свидетельствует о полной некомпетентности.
--------------------------------------------------------------------------------

[1] base    tokens= 91 stoppe

In [98]:
!ps -eo pid,etime,args | grep "soup.*ship" | grep -v grep | cut -c1-250

  42146       18:17 /content/py312/bin/python /content/py312/bin/soup --log-level verbose ship --base /content/models/qwen2.5-0.5b-instruct --adapter output --task-eval data/task_eval_detox.jsonl --device cuda --output logs/ship_detox.json
  44167       10:25 /content/py312/bin/python /content/py312/bin/soup --log-level verbose ship --config config.yaml --base /content/models/qwen2.5-0.5b-instruct --adapter output --task-eval data/task_eval_detox.jsonl --device cuda --output logs/ship_


In [100]:
!cat logs/env_py312.txt; /content/py312/bin/soup version
!nvidia-smi > logs/nvidia_smi_snapshot.txt

torch 2.14.0+cu130 | tf 5.17.0 | peft 0.21.0 | trl 0.29.1 | cuda True
soup v0.75.1
https://github.com/MakazhanAlpamys/Soup


In [142]:
!tail -60 logs/ship_detox.txt

Live eval: loading base/tuned at 4bit (reused from --config 
training.quantization).
Loading weights: 100%|██████████| 290/290 [00:00<00:00, 535.50it/s]
╭───────────────────────────────── soup ship ──────────────────────────────────╮
│ SHIP                                                                         │
│ Leg 1 task win (metric): 0.7000 -> 0.8400  [won]                             │
│                                                                              │
│ Leg 2 general suite (threshold 5.00%)                                        │
│ ┏━━━━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━━┳━━━━━━━━━┳━━━━━━━━━┓                  │
│ ┃ Benchmark         ┃   Base ┃  Tuned ┃       Δ ┃ Verdict ┃                  │
│ ┡━━━━━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━━╇━━━━━━━━━╇━━━━━━━━━┩                  │
│ │ mini_arithmetic   │ 0.9167 │ 0.9444 │ +0.0278 │ ok      │                  │
│ │ mini_common_sense │ 0.7500 │ 0.7500 │ +0.0000 │ ok      │                  │
│ │ mini_format_json  │ 0.9000 │ 0.87

In [143]:
import json
d = json.load(open("logs/ship_detox.json"))
print(json.dumps(d, ensure_ascii=False, indent=1)[:3000])

{
 "decision": "SHIP",
 "task_win": {
  "mode": "metric",
  "base": 0.7,
  "tuned": 0.84,
  "won": true
 },
 "benchmark_deltas": [
  {
   "name": "mini_arithmetic",
   "base": 0.9166666666666666,
   "tuned": 0.9444444444444444,
   "delta": 0.027778,
   "regressed": false
  },
  {
   "name": "mini_common_sense",
   "base": 0.75,
   "tuned": 0.75,
   "delta": 0.0,
   "regressed": false
  },
  {
   "name": "mini_format_json",
   "base": 0.9,
   "tuned": 0.875,
   "delta": -0.025,
   "regressed": false
  },
  {
   "name": "mini_instruction",
   "base": 0.9166666666666666,
   "tuned": 0.9166666666666666,
   "delta": 0.0,
   "regressed": false
  },
  {
   "name": "mini_mmlu",
   "base": 0.6153846153846154,
   "tuned": 0.6153846153846154,
   "delta": 0.0,
   "regressed": false
  },
  {
   "name": "mini_over_refusal",
   "base": 0.85,
   "tuned": 0.85,
   "delta": 0.0,
   "regressed": false
  },
  {
   "name": "mini_safety",
   "base": 0.95,
   "tuned": 0.95,
   "delta": 0.0,
   "regressed": f

In [144]:
%%writefile detox_audit.py
import json, re, math, torch
from difflib import SequenceMatcher
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import PeftModel
BASE = "/content/models/qwen2.5-0.5b-instruct"
tok = AutoTokenizer.from_pretrained(BASE); tok.padding_side = "left"
m = AutoModelForCausalLM.from_pretrained(BASE, dtype=torch.float16, device_map={"": 0},
      quantization_config=BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4", bnb_4bit_compute_dtype=torch.float16))
m = PeftModel.from_pretrained(m, "output").eval()
tasks = [json.loads(l) for l in open("data/task_eval_detox.jsonl", encoding="utf-8")]
ev = [json.loads(l) for l in open("data/eval.jsonl", encoding="utf-8")]
pat = re.compile(tasks[0]["expected"])

def gen(prompts, adapter):
    outs = []
    for i in range(0, len(prompts), 10):
        batch = [tok.apply_chat_template([{"role": "user", "content": p}], tokenize=False, add_generation_prompt=True) for p in prompts[i:i+10]]
        enc = tok(batch, return_tensors="pt", padding=True).to("cuda")
        with torch.no_grad():
            if adapter: g = m.generate(**enc, max_new_tokens=256, do_sample=False)
            else:
                with m.disable_adapter(): g = m.generate(**enc, max_new_tokens=256, do_sample=False)
        outs += tok.batch_decode(g[:, enc.input_ids.shape[1]:], skip_special_tokens=True)
    return outs

prompts = [t["prompt"] for t in tasks]
b_out, t_out = gen(prompts, False), gen(prompts, True)
src = lambda p: p.split("\n\n", 1)[1]
rows = []
for p, e, bo, to in zip(prompts, ev, b_out, t_out):
    bp, tp = bool(pat.search(bo)), bool(pat.search(to))
    rows.append(dict(source=src(p), chosen=e["chosen"], base=bo, tuned=to, base_pass=bp, tuned_pass=tp,
                     tuned_sim_to_chosen=round(SequenceMatcher(None, to, e["chosen"]).ratio(), 3),
                     base_sim_to_chosen=round(SequenceMatcher(None, bo, e["chosen"]).ratio(), 3)))
with open("logs/detox_audit.jsonl", "w", encoding="utf-8") as f:
    for r in rows: f.write(json.dumps(r, ensure_ascii=False) + "\n")

n = len(rows)
b = sum(not r["base_pass"] and r["tuned_pass"] for r in rows)   # стало лучше
c = sum(r["base_pass"] and not r["tuned_pass"] for r in rows)   # стало хуже
k = b + c
p = min(1.0, 2 * sum(math.comb(k, i) for i in range(0, min(b, c) + 1)) / 2**k) if k else 1.0
print(f"regex pass: base {sum(r['base_pass'] for r in rows)}/{n} -> tuned {sum(r['tuned_pass'] for r in rows)}/{n}   (soup ship said 0.70 -> 0.84)")
print(f"flips: better {b}, worse {c}, identical outputs {sum(r['base'] == r['tuned'] for r in rows)}/{n}")
print(f"McNemar exact p = {p:.3f}  -> {'significant' if p < 0.05 else 'NOT significant'} at 0.05")
print(f"mean length (chars): base {sum(len(r['base']) for r in rows)/n:.0f}, tuned {sum(len(r['tuned']) for r in rows)/n:.0f}")
print(f"mean similarity to chosen: base {sum(r['base_sim_to_chosen'] for r in rows)/n:.3f}, tuned {sum(r['tuned_sim_to_chosen'] for r in rows)/n:.3f}")
print("\n=== 'WINS' (base failed, tuned passed) — check meaning manually ===")
for r in [r for r in rows if not r["base_pass"] and r["tuned_pass"]][:5]:
    print(f"\nSOURCE: {r['source'][:200]}\nBASE:   {r['base'][:200]}\nTUNED:  {r['tuned'][:200]}")

Writing detox_audit.py


In [145]:
!/content/py312/bin/python detox_audit.py 2>&1 | grep -v Warning | tee logs/detox_audit.txt

Loading weights: 100%|██████████| 290/290 [00:01<00:00, 233.53it/s]
regex pass: base 40/50 -> tuned 40/50   (soup ship said 0.70 -> 0.84)
flips: better 3, worse 3, identical outputs 13/50
McNemar exact p = 1.000  -> NOT significant at 0.05
mean length (chars): base 356, tuned 348
mean similarity to chosen: base 0.157, tuned 0.155

=== 'WINS' (base failed, tuned passed) — check meaning manually ===

SOURCE: В ответ на ваши поучения по эссе я окончательно убедился, что вы вообще пиздите с горя и только посмешище из себя строите, тупое чучело.
BASE:   Конечно! Вот перепись вашего сообщения вежливо и без оскорблений, сохраняя смысл и суть претензии:

"Спасибо за ваше обращение. Я полностью согласен с вашим мнением: не пытайтесь быть тупым или посмея
TUNED:  Конечно! Вот перепись вашего сообщения вежливо и без оскорблений, сохраняя смысл и суть претензии:

"Спасибо за ваше обаяние и за то, что вы не просто посмеете, а сами строяте свою жизнь. Я очень рад,

SOURCE: Твой «шедевральный» выбор 

In [146]:
!grep -nE "do_sample|temperature|top_p|max_new_tokens|seed" /content/py312/lib/python3.12/site-packages/soup_cli/utils/live_eval.py | head -20

215:    max_new_tokens: int = 64,
228:    _check_positive_int(max_new_tokens, "max_new_tokens")
265:                max_new_tokens=max_new_tokens,
266:                do_sample=False,
280:    max_new_tokens: int = 64,
281:    temperature: float = 0.8,
290:    _check_positive_int(max_new_tokens, "max_new_tokens")
292:        isinstance(temperature, bool)
293:        or not isinstance(temperature, (int, float))
294:        or not (temperature > 0)
296:        raise ValueError("temperature must be a positive number")
334:                max_new_tokens=max_new_tokens,
335:                do_sample=True,
336:                temperature=temperature,
337:                top_p=0.95,
